# Pathway Moran's I + Per–Cell-Type Scoring

Standalone analysis: pick any pathway from the loaded GMT collections (hallmark, reactome, GO BP, curated), score it across all cells, compute Moran's I per tissue for treatments of interest, render a pathway × treatment heatmap, and produce a customizable cell-type × condition box plot of mean scores.

**Edit points (one per section):**
1. `pathway_panel` — which gene sets to score.
2. `analysis_treatments` — which treatments (and therefore tissues) to include.
3. `heatmap_treatments` / `heatmap_score_cols` — what to show in the heatmap.
4. `boxplot_cell_types` / `boxplot_score_col` / `boxplot_treatments` — what to show in the cell-type box plot.


# Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

import scanpy as sc
import squidpy as sq

import matplotlib.pyplot as plt
import seaborn as sns

import spatialdata as spd


# Project paths and data load

In [ ]:
zarr_file = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"

reference_dir = Path("/coh_labs/yunroseli/Jona/CAR-T/data/references")
pathway_dir = reference_dir / "pathways"

outdir = Path("/coh_labs/yunroseli/Jona/CAR-T/results/pathway_moransI_standalone")
outdir.mkdir(parents=True, exist_ok=True)

print(f"Pathway dir exists: {pathway_dir.exists()}")
print(f"Output dir: {outdir}")

sdata = spd.read_zarr(zarr_file)
adata = sdata.tables["segmentation_counts"]
print(adata)


# Normalize counts for pathway scoring

In [ ]:
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)
sc.pp.log1p(adata)

adata.uns["pathway_score_normalization"] = {
    "method": "scanpy.pp.normalize_total + scanpy.pp.log1p",
    "target_sum": 1e4,
    "use_raw": False,
}
print("Normalized adata.X with normalize_total(1e4) + log1p.")


# Load all pathway collections

Mirrors `Pathway_Visualization.ipynb`: every gene set across all four GMT files is loaded into a single `all_gene_sets` dict, with `gene_set_metadata` describing each one's collection and size.

In [ ]:
gmt_files = {
    "hallmark": pathway_dir / "mh.all.v2026.1.Mm.symbols.gmt",
    "reactome": pathway_dir / "m2.cp.reactome.v2026.1.Mm.symbols.gmt",
    "go_bp":    pathway_dir / "m5.go.bp.v2026.1.Mm.symbols.gmt",
    "curated":  pathway_dir / "m2.cp.v2026.1.Mm.symbols.gmt",
}

for name, path in gmt_files.items():
    print(name, path, path.exists())


In [ ]:
def read_gmt(gmt_path):
    gene_sets = {}
    rows = []
    with open(gmt_path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            gene_set, description = parts[0], parts[1]
            genes = parts[2:]
            gene_sets[gene_set] = genes
            rows.append({
                "gene_set": gene_set,
                "description": description,
                "n_genes": len(genes),
                "source_file": Path(gmt_path).name,
            })
    return gene_sets, pd.DataFrame(rows)


def find_pathway_names(pathways, search_term):
    if isinstance(search_term, str):
        search_term = [search_term]
    return [k for k in pathways.keys() if any(t.lower() in k.lower() for t in search_term)]


all_gene_sets = {}
gene_set_metadata_list = []
for collection, path in gmt_files.items():
    gs, meta = read_gmt(path)
    meta["collection"] = collection
    all_gene_sets.update(gs)
    gene_set_metadata_list.append(meta)

gene_set_metadata = pd.concat(gene_set_metadata_list, ignore_index=True)
print(f"Total gene sets loaded: {len(all_gene_sets):,}")
display(gene_set_metadata.groupby("collection")[["gene_set"]].count())


## Searching for pathway names

Use `find_pathway_names(all_gene_sets, ["ifn", "interferon"])` to discover the exact MSigDB names to put into `pathway_panel` below.

In [ ]:
# Example searches — edit/run as needed.
print(find_pathway_names(all_gene_sets, ["interferon_gamma"])[:10])
print(find_pathway_names(all_gene_sets, ["cytotoxicity"])[:10])
print(find_pathway_names(all_gene_sets, ["mhc"])[:10])


# Define the pathway panel and score

Left = short name (becomes the score column `<short_name>_score`).
Right = exact MSigDB gene set name.

In [ ]:
pathway_panel = {
    "IFN_gamma":              "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "MHCII_Reactome":         "REACTOME_MHC_CLASS_II_ANTIGEN_PRESENTATION",
    "MHCI_Reactome":          "REACTOME_ANTIGEN_PROCESSING_CROSS_PRESENTATION",
    "TcellCytotoxicity_GO":   "GOBP_T_CELL_MEDIATED_CYTOTOXICITY",
    "Cytolysis_GO":           "GOBP_CYTOLYSIS",
}

panel_check = []
for short_name, gene_set_name in pathway_panel.items():
    found = gene_set_name in all_gene_sets
    panel_check.append({
        "short_name": short_name,
        "gene_set_name": gene_set_name,
        "found": found,
        "n_genes": len(all_gene_sets[gene_set_name]) if found else np.nan,
        "collection": gene_set_metadata.loc[
            gene_set_metadata["gene_set"].eq(gene_set_name), "collection"
        ].iloc[0] if found else "NOT FOUND",
    })

display(pd.DataFrame(panel_check))


In [ ]:
available_genes = set(adata.var_names.astype(str))
score_records = []

for short_name, gene_set_name in pathway_panel.items():
    pathway_genes = all_gene_sets.get(gene_set_name, [])
    matched = [g for g in pathway_genes if g in available_genes]
    score_col = f"{short_name}_score"

    print(f"Scoring {short_name!r}  ({len(matched)}/{len(pathway_genes)} genes matched)...")
    if len(matched) < 5:
        print("  WARNING: too few matched genes — skipping.")
        continue

    sc.tl.score_genes(adata, gene_list=matched, score_name=score_col, use_raw=False)
    score_records.append({
        "short_name": short_name,
        "score_col": score_col,
        "gene_set_name": gene_set_name,
        "n_matched": len(matched),
        "n_total": len(pathway_genes),
    })

score_summary_df = pd.DataFrame(score_records)
score_cols = score_summary_df["score_col"].tolist()
display(score_summary_df)


# Treatments of interest → tissue list

Edit `analysis_treatments` to choose what to analyze. `analysis_tissues` is pulled from `adata.obs` so it always reflects what's actually in the dataset.

In [ ]:
analysis_treatments = [
    "CyPSCA_Tu1",
    "RTCyPSCA_Tu1",
    "RTCyPSCA_Tu2",
]

analysis_tissues = sorted(
    adata.obs.loc[adata.obs["treatment"].isin(analysis_treatments), "tissue"]
    .astype(str)
    .unique()
    .tolist()
)

print("Treatments:", analysis_treatments)
print("Tissues:")
for t in analysis_tissues:
    print(f"  - {t}")


# Moran's I per tissue per pathway

Spatial autocorrelation of each pathway score within each tissue. Tissues with fewer than 100 cells are skipped.

In [ ]:
moran_results = []

for tissue in analysis_tissues:
    print(f"\nMoran's I for tissue: {tissue}")
    ad_t = adata[adata.obs["tissue"].astype(str).eq(tissue)].copy()
    print(f"  n cells: {ad_t.n_obs:,}")

    if ad_t.n_obs < 100:
        print("  Skipping (fewer than 100 cells).")
        continue

    sq.gr.spatial_neighbors(ad_t, spatial_key="spatial", coord_type="generic", n_neighs=6)
    sq.gr.spatial_autocorr(
        ad_t,
        mode="moran",
        genes=score_cols,
        attr="obs",
        n_perms=999,
        n_jobs=8,
    )

    res = ad_t.uns["moranI"].copy().reset_index().rename(columns={"index": "score_col"})
    for col in ["tissue", "condition", "tumor_loc", "replicate_num", "treatment"]:
        res[col] = ad_t.obs[col].astype(str).iloc[0]
    res["n_cells"] = ad_t.n_obs
    res["n_neighs"] = 6
    res["n_perms"] = 999

    moran_results.append(res)

moran_df = pd.concat(moran_results, ignore_index=True)
display(moran_df.head())

moran_df.to_csv(outdir / "moranI_pathway_scores_per_tissue.csv", index=False)
print(f"\nSaved: {outdir / 'moranI_pathway_scores_per_tissue.csv'}")


# Heatmap — mean Moran's I (pathway × treatment)

Edit `heatmap_treatments` and `heatmap_score_cols` to control the matrix.

In [ ]:
heatmap_treatments = analysis_treatments
heatmap_score_cols = score_cols

heatmap_df = (
    moran_df.loc[
        moran_df["treatment"].isin(heatmap_treatments)
        & moran_df["score_col"].isin(heatmap_score_cols)
    ]
    .groupby(["score_col", "treatment"], observed=True)["I"]
    .mean()
    .unstack("treatment")
    .reindex(index=heatmap_score_cols, columns=heatmap_treatments)
)

display(heatmap_df)

fig, ax = plt.subplots(
    figsize=(1.6 * len(heatmap_treatments) + 2, 0.55 * len(heatmap_score_cols) + 1.5)
)
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".3f",
    cmap="magma",
    cbar_kws={"label": "Mean Moran's I"},
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Mean Moran's I across tissues")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

treatments_tag = "_".join(heatmap_treatments)
outfile = outdir / f"moranI_mean_heatmap_{treatments_tag}.png"
plt.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {outfile}")


# Per-cell-type box plot of mean pathway score

For the selected pathway, computes mean score per (tissue, cell type), then plots a box+strip with `x = treatment`, `y = mean_score`, `hue = cell_type`. Each dot = one tissue. Edit `boxplot_cell_types` to pick which cell types to show, and `boxplot_score_col` / `boxplot_treatments` for the rest.

In [ ]:
print("Available c2l_permissive cell types:")
print(sorted(adata.obs["c2l_permissive"].astype(str).unique().tolist()))
print("\nAvailable score columns:", score_cols)


In [ ]:
boxplot_score_col   = "IFN_gamma_score"
boxplot_treatments  = analysis_treatments
boxplot_cell_types  = ["CD8_T", "NK", "M1_like_Mac", "cDC"]

obs_sub = adata.obs.loc[
    adata.obs["treatment"].isin(boxplot_treatments)
    & adata.obs["c2l_permissive"].astype(str).isin(boxplot_cell_types)
].copy()

per_tissue = (
    obs_sub
    .groupby(["tissue", "treatment", "c2l_permissive"], observed=True)[boxplot_score_col]
    .mean()
    .reset_index()
    .rename(columns={boxplot_score_col: "mean_score"})
)

per_tissue["treatment"] = pd.Categorical(per_tissue["treatment"], categories=boxplot_treatments, ordered=True)
per_tissue["c2l_permissive"] = pd.Categorical(per_tissue["c2l_permissive"], categories=boxplot_cell_types, ordered=True)
per_tissue = per_tissue.sort_values(["treatment", "c2l_permissive"])

display(per_tissue)

fig, ax = plt.subplots(figsize=(1.8 * len(boxplot_treatments) + 2.5, 5))

sns.boxplot(
    data=per_tissue,
    x="treatment",
    y="mean_score",
    hue="c2l_permissive",
    order=boxplot_treatments,
    hue_order=boxplot_cell_types,
    showfliers=False,
    boxprops={"facecolor": "none"},
    whiskerprops={"linewidth": 1},
    ax=ax,
)
sns.stripplot(
    data=per_tissue,
    x="treatment",
    y="mean_score",
    hue="c2l_permissive",
    order=boxplot_treatments,
    hue_order=boxplot_cell_types,
    dodge=True,
    size=6,
    alpha=0.9,
    ax=ax,
)

# Deduplicate legend (boxplot + stripplot each add entries).
handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    seen.setdefault(l, h)
ax.legend(seen.values(), seen.keys(), title="Cell type", bbox_to_anchor=(1.02, 1), loc="upper left")

ax.set_xlabel("")
ax.set_ylabel(f"Mean {boxplot_score_col} per tissue")
ax.set_title(f"{boxplot_score_col} by cell type and treatment")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

cells_tag = "_".join(boxplot_cell_types)
treatments_tag = "_".join(boxplot_treatments)
outfile = outdir / f"boxplot_{boxplot_score_col}_{cells_tag}_{treatments_tag}.png"
plt.savefig(outfile, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {outfile}")
